# Script 1: Find Missing Data
This script loads a station’s raw data, scans each column for missing (NA) values, and produces a dictionary (or list) of date indices where missing data occurs. For example, you might group these by year, month, week, or specific dates.

## 1. merge raw data first

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_soil_data(station_id=1, base_dir="../datasets/TX-Data/soil_station"):
    """
    Load soil station data from .dat files.
    
    Args:
        station_id (int): Station ID (1-6).
        base_dir (str): Directory containing soil station files.
    
    Returns:
        pd.DataFrame: Raw soil data with datetime index.
    """
    file_path = Path(base_dir) / f"SM_{station_id}.dat"
    df = pd.read_csv(file_path, sep=",")

    # Convert Date to datetime format and set as index
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y %H:%M', errors='coerce')

    df = df.set_index('Date')
    
    # Remove spaces
    df.columns = df.columns.str.strip()
    
    # Convert datatype to float
    numeric_cols = ['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50', 
                    'T_5', 'T_10', 'T_20', 'T_50', 'Ppt', 'Flag']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

In [3]:
def load_met_data(station_id=1, base_dir="../datasets/TX-Data/met_station"):
    """Load and preprocess MET station data"""
    file_path = Path(base_dir) / f"MET_{station_id}.dat"
    
    # Convert Date to datetime format and set as index
    df = pd.read_csv(
        file_path,
        parse_dates=['Date'],
        index_col='Date',
        date_parser=lambda x: pd.to_datetime(x, format='%m/%d/%y %H:%M', errors='coerce'),
        dtype={
            'Ppt': np.float64,
            'Tair': np.float64,
            'RH': np.float64,
            'Wind speed': np.float64,
            'Wind direction': np.float64,
            'Srad': np.float64
        }
    )    
    # Remove spaces
    df.columns = df.columns.str.strip()
    
    # Filter data from 2015-01-01
    df = df.loc['2015-01-01 00:00:00':]

    # Convert numeric columns
    numeric_cols = ['Ppt', 'Tair', 'RH', 'Wind speed', 'Wind direction', 'Srad']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

In [4]:
def merge_raw_data(station_id):
    # Load soil and MET raw data
    df_soil = load_soil_data(station_id)
    df_met = load_met_data(station_id)
    
    # Merge on the datetime index
    merged_df = pd.merge(df_soil, df_met, how='inner', left_index=True, right_index=True, 
                         suffixes=('_soil', '_met'))
    
    # For the Ppt column, if both exist, choose one (here we choose the MET version)
    if 'Ppt_soil' in merged_df.columns and 'Ppt_met' in merged_df.columns:
        merged_df['Ppt'] = merged_df['Ppt_met']
        merged_df.drop(columns=['Ppt_soil', 'Ppt_met'], inplace=True)
        
    return merged_df

In [5]:
def save_merged_data(df, station_id, output_dir="raw_merged_data"):
    """
    Save merged data to CSV.
    
    Args:
        df (pd.DataFrame): Merged soil and met data.
        station_id (int): Station ID (1-6).
        output_dir (str): Output directory path.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/raw_merged_station_{station_id}.csv"
    df.to_csv(output_path)
    print(f"Saved merged data to: {output_path}")
    return df

In [6]:
save_merged_data(merge_raw_data(1), 1)
save_merged_data(merge_raw_data(2), 2)
save_merged_data(merge_raw_data(3), 3)
save_merged_data(merge_raw_data(4), 4)
save_merged_data(merge_raw_data(5), 5)
save_merged_data(merge_raw_data(6), 6)

Saved merged data to: raw_merged_data/raw_merged_station_1.csv
Saved merged data to: raw_merged_data/raw_merged_station_2.csv
Saved merged data to: raw_merged_data/raw_merged_station_3.csv
Saved merged data to: raw_merged_data/raw_merged_station_4.csv
Saved merged data to: raw_merged_data/raw_merged_station_5.csv
Saved merged data to: raw_merged_data/raw_merged_station_6.csv


,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Tair,RH,Wind speed,Wind direction,Srad,Ppt
Date,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.116,0.123,0.137,0.084,3.85,4.64,5.82,9.58,0,-1.013,82.40,0.995,36.94,0.00,0.0
2015-01-01 01:00:00,0.116,0.122,0.137,0.085,3.84,4.62,5.78,9.52,768,-0.981,83.30,0.857,30.15,0.56,0.0
2015-01-01 02:00:00,0.116,0.122,0.137,0.085,3.83,4.59,5.74,9.47,768,-0.910,83.90,1.040,44.80,0.00,0.0
2015-01-01 03:00:00,0.115,0.122,0.137,0.085,3.84,4.57,5.69,9.43,768,-0.842,82.60,0.936,50.38,0.00,0.0
2015-01-01 04:00:00,0.115,0.123,0.137,0.085,3.84,4.55,5.64,9.39,0,-0.785,89.90,0.606,358.20,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-31 20:00:00,0.104,0.101,0.090,0.062,35.04,34.77,33.60,30.74,0,31.410,100.00,0.122,181.60,0.02,0.0
2021-08-31 21:00:00,0.104,0.100,0.090,0.062,34.52,34.46,33.64,30.80,0,30.310,100.00,0.013,184.30,0.00,0.0
2021-08-31 22:00:00,0.103,0.100,0.090,0.062,33.99,34.07,33.58,30.86,0,29.990,2.55,0.309,168.60,0.00,0.0


## 2. find missing values

In [7]:
def find_missing_data(df):
    missing_info = {}
    for col in df.columns:
        missing_dates = df.index[df[col].isnull()].tolist()
        if missing_dates:
            missing_info[col] = missing_dates
    return missing_info

df_merged = pd.read_csv("raw_merged_data/raw_merged_station_1.csv", index_col=0, parse_dates=True)
missing_info = find_missing_data(df_merged)
print("Missing data (per column):")
for col, dates in missing_info.items():
    print(f"{col}: {dates}")

Missing data (per column):
SWC_5: [Timestamp('2017-05-27 16:00:00'), Timestamp('2017-05-27 17:00:00'), Timestamp('2017-05-27 18:00:00'), Timestamp('2017-05-27 19:00:00'), Timestamp('2017-05-27 20:00:00'), Timestamp('2017-05-27 21:00:00'), Timestamp('2017-05-27 22:00:00'), Timestamp('2017-05-28 11:00:00'), Timestamp('2017-05-28 12:00:00'), Timestamp('2017-05-28 13:00:00'), Timestamp('2017-05-28 14:00:00'), Timestamp('2017-05-28 15:00:00'), Timestamp('2017-05-28 16:00:00'), Timestamp('2017-05-28 17:00:00'), Timestamp('2017-05-28 18:00:00'), Timestamp('2017-05-28 19:00:00'), Timestamp('2017-06-08 16:00:00'), Timestamp('2017-06-08 17:00:00'), Timestamp('2017-06-08 18:00:00'), Timestamp('2017-06-08 19:00:00'), Timestamp('2017-06-08 20:00:00'), Timestamp('2017-06-08 21:00:00'), Timestamp('2017-06-08 22:00:00'), Timestamp('2017-06-08 23:00:00'), Timestamp('2017-06-09 00:00:00'), Timestamp('2017-06-09 01:00:00'), Timestamp('2017-06-09 02:00:00'), Timestamp('2017-06-09 03:00:00'), Timestamp('20